# Duplicate map records

One edit to a JupyterGIS map is recorded once for every copy of the map in the browser. A map shown in a notebook's output is a second copy of a map that is already open in its own tab, so one browser window is enough to cause it.

Start JupyterLab and Trail as in [setup](setup.md), then open this notebook in `trail-bugs`.

| | Edit from Python | Edit in the map tab |
|---|---|---|
| **Expected** | one source and one layer record | one layer record |
| **Observed** | two source and two layer records, all `local: False` | two layer records, `local: True` and `local: False` |

Example records come from a run on 14 November 2026 that made the same edits with a script. Ids and times will differ. Event times print in UTC.

## Why it happens

A map is a shared document. Jupyter Server keeps it in memory and writes it to the `.jGIS` file. Each view of the map in the browser keeps its own synced copy: the map tab is one copy, and a map shown in a notebook's output is another. The kernel that runs `GISDocument` keeps a copy too.

The JupyterGIS event feed runs in the browser. It reports each change to each browser copy, so a change that reaches two copies is reported twice. `local` says whether the copy reporting the change is the one where it was made. The kernel has no event feed, so its own copy reports nothing.

## Open the map

Open `maps/earthquakes.jGIS` from the file browser. Drag this notebook's tab to the right of the map so both are visible.

Trail records the map as `opened`. The browser now holds one copy of the map: the map tab.

```text
6. DocumentV1Event
    id: 706ae3b3795145e0b1daf52f44e910d7
    timestamp: 03:15:52.989
    schema_id: https://schema.jupytergis.org/events/document/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: opened
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
```

## Connect from Python

In [ ]:
from jupytergis import GISDocument

doc = GISDocument('maps/earthquakes.jGIS')

`doc` is the kernel's copy of the map. It is not shown in the notebook, so the browser still holds one copy.

## Edit from Python with one copy

In [ ]:
doc.add_geojson_layer(path='data/crossings.geojson', name='one copy')

This is the expected behavior. The map tab reports the new source and layer once each. Both are `local: False`, since the edit came from the kernel.

```text
18. SourceV1Event
    id: 51256192398b4474b6ff140d3720846f
    timestamp: 03:16:04.766
    schema_id: https://schema.jupytergis.org/events/source/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: added
    local: False
    source_id: 229be65b-4343-4cd8-8c98-92b12c720252
    new: GeoJSONSource inline data 374 B
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
19. LayerV1Event
    id: 7c6ec93de2e148ea8168855f911707ac
    timestamp: 03:16:04.788
    schema_id: https://schema.jupytergis.org/events/layer/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: added
    local: False
    layer_id: 30257e75-765e-419e-8edf-02e9554b9312
    layer: one copy
    source: GeoJSONSource inline data 374 B
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
```

## Show the map in the notebook

In [ ]:
doc

The cell output draws the map. This is a second copy in the browser. Trail records the map as `opened` again, although its tab never closed.

```text
24. DocumentV1Event
    id: 7a1a3b3fdc974a4b90274b909598c847
    timestamp: 03:16:10.075
    schema_id: https://schema.jupytergis.org/events/document/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: opened
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
```

## Edit from Python with two copies

In [ ]:
doc.add_geojson_layer(path='data/crossings.geojson', name='two copies')

The same edit as before now gives four records. The map tab and the notebook output each report the source and the layer. Each pair has the same `source_id` or `layer_id`, and every record is `local: False`.

```text
28. SourceV1Event
    id: ec557f9d94ca4de0a1e22a88f93e53c4
    timestamp: 03:16:15.226
    schema_id: https://schema.jupytergis.org/events/source/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: added
    local: False
    source_id: 7140a818-5d08-497f-9778-7b5e80b01d88
    new: GeoJSONSource inline data 374 B
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
29. LayerV1Event
    id: b147466ac78c4476b580e9546b25b32a
    timestamp: 03:16:15.235
    schema_id: https://schema.jupytergis.org/events/layer/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: added
    local: False
    layer_id: dd939578-364b-426a-8108-d25f7ce3775d
    layer: two copies
    source: GeoJSONSource inline data 374 B
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
30. SourceV1Event
    id: 60e5a4d1fda847d6a965e8fee1d7b570
    timestamp: 03:16:15.433
    schema_id: https://schema.jupytergis.org/events/source/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: added
    local: False
    source_id: 7140a818-5d08-497f-9778-7b5e80b01d88
    new: GeoJSONSource inline data 374 B
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
31. LayerV1Event
    id: 136a7111ade74e87ab2dc874334a9c6e
    timestamp: 03:16:15.455
    schema_id: https://schema.jupytergis.org/events/layer/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: added
    local: False
    layer_id: dd939578-364b-426a-8108-d25f7ce3775d
    layer: two copies
    source: GeoJSONSource inline data 374 B
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
```

The map file changed once, so its change record appears once:

```text
33. MetadataEvent
    id: 01f6db54fc2f47b8aa20a2ce606e4934
    timestamp: 03:16:17.436
    src_path: maps/earthquakes.jGIS
    format: jgis
    basis: watchdog
    st_size: 9.7 KiB
    st_mtime: 2026-11-14 19:16:16
    change: layer 'two copies' added
    change: source 'two copies Source' added
    change: layer order changed
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
```

## Edit in the map tab

In the map tab, right-click **Earthquakes** in the **Layers** panel, choose **Layer Properties** and set **Opacity** to 0.4. The dialog applies each keystroke, so typing can record more than one pair.

The map tab reports the change as `local: True` and the notebook output reports it again as `local: False`:

```text
34. LayerV1Event
    id: ba2f1387782c4d81a1c778353cb0a81f
    timestamp: 03:16:20.471
    schema_id: https://schema.jupytergis.org/events/layer/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: updated
    local: True
    layer_id: ccf98385-bbe3-425b-9dde-fa64d31c196a
    layer: Earthquakes
    change: opacity 1 → 0.4
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
35. LayerV1Event
    id: df6f5a42f4944755adcc9748371fd804
    timestamp: 03:16:20.948
    schema_id: https://schema.jupytergis.org/events/layer/v1
    server: http://localhost:8897/
    src_path: maps/earthquakes.jGIS
    verb: updated
    local: False
    layer_id: ccf98385-bbe3-425b-9dde-fa64d31c196a
    layer: Earthquakes
    change: opacity 1 → 0.4
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
```

The map file's change record again appears once:

```text
36. MetadataEvent
    id: 6ed28e167bfe405289e1c001d61837b8
    timestamp: 03:16:22.656
    src_path: maps/earthquakes.jGIS
    format: jgis
    basis: watchdog
    st_size: 9.7 KiB
    st_mtime: 2026-11-14 19:16:21
    change: layer 'Earthquakes' opacity 1.0 → 0.4
    entry: e0ea0a999f7d47fc807e4fc8f1ed7b20
```

## Count the records

Enter these in Trail:

| Command | Shows | In this run |
|---|---|---|
| `events cls=LayerEvent or cls=SourceEvent` | every source and layer record | 8 records for 3 edits: 18, 19, 28–31, 34, 35. 5 expected |
| `events cls=LayerEvent is_local=False` | layer records from a copy that did not make the change | 19, 29, 31, 35 |
| `events cls=DocumentEvent verb=opened` | each time a copy of the map was opened | 6 and 24 |

## Why `local` alone does not fix it

| Edit made in | Map tab reports | Notebook output reports |
|---|---|---|
| the map tab | `local: True` | `local: False` |
| Python, `doc` shown | `local: False` | `local: False` |
| Python, `doc` not shown | `local: False` | not shown, no report |

Keeping `local: True` and dropping the `local: False` copies would leave one record for edits in the map tab. Edits from Python have no `local: True` copy, so the same rule would drop them all. Trail would have to match the copies by map, source or layer id and change within a short time.

The same duplicates appear with the map open in two browser windows. Two people editing one map should duplicate the same way, since each person's browser has its own copy. That case is not tested.

## Where in the code

| Where | What it does |
|---|---|
| `eventfeed/python/jupytergis_lab/src/notebookrenderer.ts:146` | Adds the notebook output's map to the same tracker as map tabs. |
| `eventfeed/python/jupytergis_core/src/eventFeed/plugin.ts:174` | Wires the event feed once per copy. Two tabs that share a copy report once; the notebook output has its own copy. |
| `eventfeed/packages/schema/src/doc.ts:610` | Sets `local` from the Yjs transaction. |
| `trail/src/trail/jupyter/event.py:275` | `trail-todo: origin`: copies are recorded, not dropped. |

Paths are from `stable` of both repositories.